# 05_outliers_and_split.ipynb

## Σκοπός

Το `NB05` είναι αυστηρά περιορισμένο σε τρεις μόνο ρόλους:

1. να διαβάσει το canonical engineered dataset από το `NB04`
2. να εφαρμόσει **leakage-safe outlier handling**
3. να κατασκευάσει και να εξάγει τα canonical:
   - `train_final.csv`
   - `val_final.csv`
   - `test_final.csv`

## Τι δεν κάνει αυτό το notebook

Το `NB05` **δεν είναι**:

- feature engineering notebook
- modeling notebook
- diagnostics notebook
- hyperparameter tuning notebook

## Canonical upstream contract

Το `NB05` πρέπει να καταναλώνει **μόνο** το canonical downstream artifact του `NB04`:

- `final_feature_engineered_dataset.csv`

Δεν επιτρέπεται:

- raw rediscovery
- loose reparsing raw timestamps
- upstream logic duplication από `NB02` ή `NB04`

## Split philosophy

Το current canonical split του project είναι πλέον **flag-aware temporal split**:

- `test_df = test_flag == 1`
- `pretest_df = test_flag == 0`
- `val_df` = το τελικό χρονικό tail του pre-test window
- `train_df` = το υπόλοιπο pre-test window

## Outlier handling philosophy

Το clipping του target πρέπει να είναι **train-aware**:

- thresholds fit μόνο στο `train`
- εφαρμογή των ίδιων thresholds σε `train / val / test`

Έτσι αποφεύγεται leakage από `validation` ή `test` προς το preprocessing.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import csv

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# Βοηθητικές συναρτήσεις path handling
# ============================================================

def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το project root ανεβαίνοντας από το current working directory.
    Απαιτούμε να υπάρχουν τουλάχιστον οι φάκελοι data/ και notebooks/.
    """
    current = start_path.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        "Δεν βρέθηκε το project root με φακέλους 'data/' και 'notebooks/'."
    )


def resolve_under_root(path_like: str | Path, root: Path) -> Path:
    """
    Αν το path είναι σχετικό, το λύνουμε κάτω από το project root.
    Αν είναι absolute, το κρατάμε ως έχει.
    """
    p = Path(path_like)
    return p if p.is_absolute() else root / p


# ============================================================
# Project root / imports
# ============================================================

ROOT = find_project_root(Path.cwd())

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import src.config as cfg


# ============================================================
# Canonical paths / βασικές στήλες
# ============================================================

FINAL_DATASET = resolve_under_root(
    getattr(cfg, "FINAL_DATASET", "data/processed/final_feature_engineered_dataset.csv"),
    ROOT,
)

DATA_PROCESSED = resolve_under_root(
    getattr(cfg, "DATA_PROCESSED", "data/processed"),
    ROOT,
)

PARK_ID_COLUMN = getattr(cfg, "PARK_ID_COLUMN", "park_id")
TIMESTAMP_COLUMN = getattr(cfg, "TIMESTAMP_COLUMN", "timestamp")
TARGET_COLUMN = getattr(cfg, "TARGET_COLUMN", "Power_Output_Normalized")
BASELINE_COLUMN = getattr(cfg, "BASELINE_COLUMN", "Baseline_Prediction")

TEST_FLAG_COLUMN = "test_flag"

# Χρησιμοποιούμε validation tail 30 ημερών πριν από το πρώτο test timestamp
VALIDATION_HORIZON_DAYS = 30
VALIDATION_HORIZON = pd.Timedelta(days=VALIDATION_HORIZON_DAYS)

# Στήλες που ΔΕΝ πρέπει by default να περάσουν στα downstream models
CONTROL_IDENTIFIER_COLUMNS = [
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
]

PROVENANCE_AUDIT_COLUMNS = [
    TEST_FLAG_COLUMN,
    BASELINE_COLUMN,
]

# Raw/helper/audit στήλες που δεν πρέπει να εμφανίζονται στο NB05 input
RAW_AUDIT_COLUMNS_SHOULD_NOT_REACH_NB05 = [
    "fcst_time",
    "time",
    "loader_input_sep_used",
    "loader_target_sep_used",
    "loader_input_ts_format",
    "loader_target_ts_format",
    "loader_scaled_input_by_1000",
    "num_train_samples",
    "num_test_samples",
]

print("Project root:", ROOT)
print("FINAL_DATASET:", FINAL_DATASET)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("PARK_ID_COLUMN:", PARK_ID_COLUMN)
print("TIMESTAMP_COLUMN:", TIMESTAMP_COLUMN)
print("TARGET_COLUMN:", TARGET_COLUMN)
print("BASELINE_COLUMN:", BASELINE_COLUMN)
print("TEST_FLAG_COLUMN:", TEST_FLAG_COLUMN)
print("VALIDATION_HORIZON_DAYS:", VALIDATION_HORIZON_DAYS)

Project root: C:\Users\diony\Desktop\WindPower_DigitalTwin
FINAL_DATASET: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\final_feature_engineered_dataset.csv
DATA_PROCESSED: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed
PARK_ID_COLUMN: park_id
TIMESTAMP_COLUMN: timestamp
TARGET_COLUMN: Power_Output_Normalized
BASELINE_COLUMN: Baseline_Prediction
TEST_FLAG_COLUMN: test_flag
VALIDATION_HORIZON_DAYS: 30


## Εισαγωγή canonical dataset και fail-fast input contract

Σε αυτό το βήμα ελέγχεται αυστηρά ότι:

- υπάρχει το `final_feature_engineered_dataset.csv`
- το dataset περιέχει τα απαιτούμενα backbone columns
- το `timestamp` backbone γίνεται parse με explicit format
- δεν υπάρχουν duplicate `(park_id, timestamp)` rows
- δεν έχουν διαρρεύσει raw audit/helper columns από upstream στάδια
- η χρονική σειρά είναι monotonic ανά `park_id`

Αν κάποιο από τα παραπάνω αποτύχει, το notebook σταματά αμέσως.

In [2]:
# ============================================================
# Canonical input existence checks
# ============================================================

if not FINAL_DATASET.exists():
    raise FileNotFoundError(
        f"Λείπει το canonical NB04 export: {FINAL_DATASET}"
    )

if FINAL_DATASET.name != "final_feature_engineered_dataset.csv":
    raise ValueError(
        "Το NB05 πρέπει να διαβάζει μόνο το canonical "
        "`final_feature_engineered_dataset.csv`."
    )


# ============================================================
# Safe load
# ============================================================

dtype_map = {
    PARK_ID_COLUMN: "string",
    TEST_FLAG_COLUMN: "Int64",
}

df = pd.read_csv(FINAL_DATASET, dtype=dtype_map)

required_columns = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TARGET_COLUMN,
    TEST_FLAG_COLUMN,
    BASELINE_COLUMN,
}

missing_required = sorted(required_columns - set(df.columns))
if missing_required:
    raise KeyError(f"Λείπουν required columns: {missing_required}")


# ============================================================
# Duplicate column-name check
# ============================================================

if df.columns.duplicated().any():
    duplicated_names = df.columns[df.columns.duplicated()].tolist()
    raise ValueError(
        f"Βρέθηκαν duplicate column names στο input: {duplicated_names}"
    )


# ============================================================
# Explicit timestamp parse
# Σημείωση:
# Το NB05 δεν ξανακάνει raw parsing. Εδώ κάνουμε parse μόνο
# του canonical downstream timestamp backbone με strict format.
# ============================================================

df[TIMESTAMP_COLUMN] = pd.to_datetime(
    df[TIMESTAMP_COLUMN],
    format="%Y-%m-%d %H:%M:%S",
    errors="raise",
)


# ============================================================
# Έλεγχος για upstream raw/helper leakage
# ============================================================

unexpected_audit_columns = [
    col for col in RAW_AUDIT_COLUMNS_SHOULD_NOT_REACH_NB05 if col in df.columns
]

if unexpected_audit_columns:
    raise ValueError(
        "Βρέθηκαν upstream raw/helper columns που δεν πρέπει να φτάνουν στο NB05: "
        f"{unexpected_audit_columns}"
    )


# ============================================================
# Core null checks
# ============================================================

core_nulls = df[
    [PARK_ID_COLUMN, TIMESTAMP_COLUMN, TARGET_COLUMN, TEST_FLAG_COLUMN]
].isnull().sum()

if int(core_nulls.sum()) > 0:
    raise ValueError(
        "Βρέθηκαν null values στα core columns:\n"
        f"{core_nulls.to_string()}"
    )


# ============================================================
# Deterministic ordering
# ============================================================

df = (
    df.sort_values([PARK_ID_COLUMN, TIMESTAMP_COLUMN], kind="mergesort")
      .reset_index(drop=True)
)


# ============================================================
# Duplicate backbone key check
# ============================================================

duplicate_key_count = int(
    df.duplicated(subset=[PARK_ID_COLUMN, TIMESTAMP_COLUMN]).sum()
)

if duplicate_key_count > 0:
    raise ValueError(
        f"Βρέθηκαν duplicate ({PARK_ID_COLUMN}, {TIMESTAMP_COLUMN}) rows: "
        f"{duplicate_key_count}"
    )


# ============================================================
# Monotonic temporal ordering ανά park
# ============================================================

non_monotonic_parks = []

for park_id, park_slice in df.groupby(PARK_ID_COLUMN, sort=False):
    if not park_slice[TIMESTAMP_COLUMN].is_monotonic_increasing:
        non_monotonic_parks.append(park_id)

if non_monotonic_parks:
    raise ValueError(
        "Βρέθηκαν parks με μη monotonic temporal ordering: "
        f"{non_monotonic_parks[:10]}"
    )

print("Canonical input contract passed")
print("-" * 70)
print(f"Rows          : {len(df):,}")
print(f"Columns       : {len(df.columns):,}")
print(f"Unique parks  : {df[PARK_ID_COLUMN].nunique():,}")
print(f"Min timestamp : {df[TIMESTAMP_COLUMN].min()}")
print(f"Max timestamp : {df[TIMESTAMP_COLUMN].max()}")

display(df.head())

Canonical input contract passed
----------------------------------------------------------------------
Rows          : 3,252,070
Columns       : 47
Unique parks  : 256
Min timestamp : 2018-12-08 06:00:00
Max timestamp : 2020-06-01 23:00:00


,park_id,timestamp,test_flag,Power_Output_Normalized,Baseline_Prediction,turbine,hub_height_m,rotor_diameter_m,nominal_power_kW,lat,...,Power_Output_Normalized_lag_1,Power_Output_Normalized_lag_3,Power_Output_Normalized_lag_6,Wind_Speed_100m_ms_lag_1,Wind_Speed_100m_ms_lag_3,Wind_Speed_100m_ms_lag_6,power_rolling_mean_6h,power_rolling_std_6h,wind_rolling_mean_6h,wind_rolling_std_6h
0,00011,2018-12-08 06:00:00,0,0.134,0.790,"E-82 E4 (2,35MW)",84,82.0,2350.0,48.0,...,0.265,0.248,0.100,12.342724,13.399685,12.191844,0.200500,0.069356,12.955831,0.727989
1,00011,2018-12-08 07:00:00,0,0.117,0.694,"E-82 E4 (2,35MW)",84,82.0,2350.0,48.0,...,0.134,0.232,0.126,12.053271,14.026279,13.298313,0.206167,0.060301,12.932736,0.758627
2,00011,2018-12-08 08:00:00,0,0.330,0.605,"E-82 E4 (2,35MW)",84,82.0,2350.0,48.0,...,0.117,0.265,0.232,11.243089,12.342724,12.476142,0.204667,0.062756,12.590198,0.989428
3,00011,2018-12-08 09:00:00,0,0.314,0.759,"E-82 E4 (2,35MW)",84,82.0,2350.0,48.0,...,0.330,0.134,0.248,10.625930,12.053271,13.399685,0.221000,0.081304,12.281830,1.278252
4,00011,2018-12-08 10:00:00,0,0.367,0.790,"E-82 E4 (2,35MW)",84,82.0,2350.0,48.0,...,0.314,0.117,0.232,11.788309,11.243089,14.026279,0.232000,0.089717,12.013267,1.160245


## Ρητός διαχωρισμός column roles

Για να μη διαρρεύσουν leakage-prone ή helper columns προς τα `NB06` και `NB07`,
οι στήλες χωρίζονται ρητά σε τέσσερις κατηγορίες:

1. **target**
2. **control / identifier**
3. **candidate modeling features**
4. **provenance / audit**

Επιπλέον, κρατάμε και το πρακτικό subset:

- `NUMERIC_MODELING_FEATURES`

ώστε τα downstream tabular baselines να μην πάρουν κατά λάθος string/object columns
χωρίς ρητή encoder policy.

In [3]:
# ============================================================
# Column taxonomy
# ============================================================

candidate_modeling_features = [
    col
    for col in df.columns
    if col not in set(
        [TARGET_COLUMN] + CONTROL_IDENTIFIER_COLUMNS + PROVENANCE_AUDIT_COLUMNS
    )
]

numeric_modeling_features = [
    col for col in candidate_modeling_features
    if pd.api.types.is_numeric_dtype(df[col])
]

non_numeric_candidate_features = [
    col for col in candidate_modeling_features
    if col not in numeric_modeling_features
]


def classify_column(col: str) -> str:
    if col == TARGET_COLUMN:
        return "target"
    if col in CONTROL_IDENTIFIER_COLUMNS:
        return "control_identifier"
    if col in PROVENANCE_AUDIT_COLUMNS:
        return "provenance_audit"
    return "candidate_modeling_feature"


column_role_df = pd.DataFrame(
    {
        "column": df.columns,
        "role": [classify_column(col) for col in df.columns],
        "dtype": [str(dtype) for dtype in df.dtypes],
        "numeric_candidate": [col in numeric_modeling_features for col in df.columns],
    }
)

TARGET_COLUMNS = [TARGET_COLUMN]
CANDIDATE_MODELING_FEATURES = candidate_modeling_features
NUMERIC_MODELING_FEATURES = numeric_modeling_features

print("Column taxonomy")
print("-" * 70)
print("TARGET_COLUMNS:")
print(TARGET_COLUMNS)
print()

print("CONTROL_IDENTIFIER_COLUMNS:")
print(CONTROL_IDENTIFIER_COLUMNS)
print()

print("PROVENANCE_AUDIT_COLUMNS:")
print(PROVENANCE_AUDIT_COLUMNS)
print()

print(f"CANDIDATE_MODELING_FEATURES ({len(CANDIDATE_MODELING_FEATURES)}):")
print(CANDIDATE_MODELING_FEATURES)
print()

print(f"NUMERIC_MODELING_FEATURES ({len(NUMERIC_MODELING_FEATURES)}):")
print(NUMERIC_MODELING_FEATURES)
print()

if non_numeric_candidate_features:
    print("Non-numeric candidate features (χρειάζονται explicit encoding policy):")
    print(non_numeric_candidate_features)

display(column_role_df)

Column taxonomy
----------------------------------------------------------------------
TARGET_COLUMNS:
['Power_Output_Normalized']

CONTROL_IDENTIFIER_COLUMNS:
['park_id', 'timestamp']

PROVENANCE_AUDIT_COLUMNS:
['test_flag', 'Baseline_Prediction']

CANDIDATE_MODELING_FEATURES (42):
['turbine', 'hub_height_m', 'rotor_diameter_m', 'nominal_power_kW', 'lat', 'long', 'nwp_fcst_horiz_hours', 'T_HAG_2_M', 'RELHUM_HAG_2_M', 'PS_SFC_0_M', 'U_GVL_58_HL', 'V_GVL_58_HL', 'U_GVL_60_HL', 'V_GVL_60_HL', 'ASWDIFDS_SFC_0_M', 'ASWDIRS_SFC_0_M', 'U_GVL_58_HL_m1', 'V_GVL_58_HL_m1', 'U_GVL_60_HL_m1', 'V_GVL_60_HL_m1', 'U_GVL_58_HL_p1', 'V_GVL_58_HL_p1', 'U_GVL_60_HL_p1', 'V_GVL_60_HL_p1', 'ws_ref', 'Wind_Speed_100m_ms', 'hour', 'month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'Power_Output_Normalized_lag_1', 'Power_Output_Normalized_lag_3', 'Power_Output_Normalized_lag_6', 'Wind_Speed_100m_ms_lag_1', 'Wind_Speed_100m_ms_lag_3', 'Wind_Speed_100m_ms_lag_6', 'power_rolling_mean_6h', 'power_rolling

,column,role,dtype,numeric_candidate
0,park_id,control_identifier,string,False
1,timestamp,control_identifier,datetime64[us],False
2,test_flag,provenance_audit,Int64,False
3,Power_Output_Normalized,target,float64,False
4,Baseline_Prediction,provenance_audit,float64,False
5,turbine,candidate_modeling_feature,str,False
6,hub_height_m,candidate_modeling_feature,int64,True
7,rotor_diameter_m,candidate_modeling_feature,float64,True
8,nominal_power_kW,candidate_modeling_feature,float64,True
9,lat,candidate_modeling_feature,float64,True


## Split-semantics audit

Πριν γίνει το split, το notebook ελέγχει ότι το `test_flag` συμπεριφέρεται ως πραγματικό
predefined test marker:

- υπάρχει και είναι binary
- υπάρχουν και pre-test και flagged-test samples
- το flagged segment είναι χρονικά μεταγενέστερο
- το `test_flag` δεν “γυρίζει” από 1 πίσω σε 0 μέσα στο ίδιο park

Μετά:

- `test_df` = όλα τα `test_flag == 1`
- `pretest_df` = όλα τα `test_flag == 0`
- `val_df` = το τελικό χρονικό tail του pre-test window
- `train_df` = το παλαιότερο pre-test τμήμα

Σημαντικό:
η λέξη-κλειδί εδώ είναι **temporal tail**, όχι strong claim τύπου “contiguous block”.

In [4]:
# ============================================================
# test_flag integrity checks
# ============================================================

observed_flag_values = sorted(df[TEST_FLAG_COLUMN].dropna().unique().tolist())

if observed_flag_values != [0, 1]:
    raise ValueError(
        f"Το {TEST_FLAG_COLUMN} πρέπει να είναι binary [0, 1]. "
        f"Observed={observed_flag_values}"
    )

pretest_df = df.loc[df[TEST_FLAG_COLUMN] == 0].copy()
test_df = df.loc[df[TEST_FLAG_COLUMN] == 1].copy()

if pretest_df.empty:
    raise ValueError("Δεν βρέθηκαν pre-test rows με test_flag == 0.")

if test_df.empty:
    raise ValueError("Δεν βρέθηκαν test rows με test_flag == 1.")

first_test_timestamp = test_df[TIMESTAMP_COLUMN].min()
last_pretest_timestamp = pretest_df[TIMESTAMP_COLUMN].max()

if not last_pretest_timestamp < first_test_timestamp:
    raise ValueError(
        "Το pre-test window δεν τελειώνει αυστηρά πριν από το test window. "
        f"last_pretest={last_pretest_timestamp}, first_test={first_test_timestamp}"
    )


# ============================================================
# Per-park monotonic test_flag behavior
# Δεν επιτρέπουμε pattern 0 -> 1 -> 0 μέσα στο ίδιο park.
# ============================================================

flag_regression_parks = []

for park_id, park_slice in df.groupby(PARK_ID_COLUMN, sort=False):
    flag_values = park_slice[TEST_FLAG_COLUMN].astype(int).to_numpy()

    if np.any(np.diff(flag_values) < 0):
        flag_regression_parks.append(park_id)

if flag_regression_parks:
    raise ValueError(
        "Βρέθηκαν parks όπου το test_flag regress από 1 πίσω σε 0: "
        f"{flag_regression_parks[:10]}"
    )


# ============================================================
# Validation tail μέσα στο pre-test window
# ============================================================

validation_start_timestamp = first_test_timestamp - VALIDATION_HORIZON

if validation_start_timestamp <= pretest_df[TIMESTAMP_COLUMN].min():
    raise ValueError(
        "Το validation horizon είναι υπερβολικά μεγάλο για το διαθέσιμο pre-test window. "
        f"validation_start={validation_start_timestamp}, "
        f"pretest_min={pretest_df[TIMESTAMP_COLUMN].min()}"
    )

train_df = pretest_df.loc[
    pretest_df[TIMESTAMP_COLUMN] < validation_start_timestamp
].copy()

val_df = pretest_df.loc[
    pretest_df[TIMESTAMP_COLUMN] >= validation_start_timestamp
].copy()

if train_df.empty:
    raise ValueError("Το train split βγήκε κενό.")

if val_df.empty:
    raise ValueError("Το validation split βγήκε κενό.")


# ============================================================
# Temporal boundary checks
# ============================================================

if not train_df[TIMESTAMP_COLUMN].max() < val_df[TIMESTAMP_COLUMN].min():
    raise ValueError("Βρέθηκε temporal overlap μεταξύ train και val.")

if not val_df[TIMESTAMP_COLUMN].max() < test_df[TIMESTAMP_COLUMN].min():
    raise ValueError("Βρέθηκε temporal overlap μεταξύ val και test.")


# ============================================================
# Exact key overlap checks
# ============================================================

def assert_no_key_overlap(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_name: str,
    right_name: str,
) -> None:
    overlap = left_df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN]].merge(
        right_df[[PARK_ID_COLUMN, TIMESTAMP_COLUMN]],
        on=[PARK_ID_COLUMN, TIMESTAMP_COLUMN],
        how="inner",
    )

    if not overlap.empty:
        raise ValueError(
            f"Βρέθηκε overlap μεταξύ {left_name} και {right_name}: "
            f"{len(overlap):,} rows"
        )

assert_no_key_overlap(train_df, val_df, "train", "val")
assert_no_key_overlap(train_df, test_df, "train", "test")
assert_no_key_overlap(val_df, test_df, "val", "test")


# ============================================================
# Reconstruction check
# ============================================================

reconstructed_rows = len(train_df) + len(val_df) + len(test_df)

if reconstructed_rows != len(df):
    raise ValueError(
        "Το split reconstruction δεν επιστρέφει το αρχικό row count. "
        f"original={len(df):,}, reconstructed={reconstructed_rows:,}"
    )


split_summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "unique_parks": [
            train_df[PARK_ID_COLUMN].nunique(),
            val_df[PARK_ID_COLUMN].nunique(),
            test_df[PARK_ID_COLUMN].nunique(),
        ],
        "min_timestamp": [
            train_df[TIMESTAMP_COLUMN].min(),
            val_df[TIMESTAMP_COLUMN].min(),
            test_df[TIMESTAMP_COLUMN].min(),
        ],
        "max_timestamp": [
            train_df[TIMESTAMP_COLUMN].max(),
            val_df[TIMESTAMP_COLUMN].max(),
            test_df[TIMESTAMP_COLUMN].max(),
        ],
        "observed_flags": [
            sorted(train_df[TEST_FLAG_COLUMN].dropna().unique().tolist()),
            sorted(val_df[TEST_FLAG_COLUMN].dropna().unique().tolist()),
            sorted(test_df[TEST_FLAG_COLUMN].dropna().unique().tolist()),
        ],
    }
)

print("Flag-aware split completed successfully")
print("-" * 70)
display(split_summary_df)

Flag-aware split completed successfully
----------------------------------------------------------------------


,split,rows,unique_parks,min_timestamp,max_timestamp,observed_flags
0,train,1982736,256,2018-12-08 06:00:00,2019-11-01 00:00:00,[0]
1,val,182998,256,2019-11-01 01:00:00,2019-12-01 00:00:00,[0]
2,test,1086336,256,2019-12-01 01:00:00,2020-06-01 23:00:00,[1]


## Outlier handling policy

Το outlier handling στο `NB05` είναι αυστηρά **train-aware**.

Εδώ εφαρμόζουμε:

- outlier fitting μόνο στο **train split**
- Z-score clipping μόνο για το **target**
- εφαρμογή των ίδιων thresholds σε:
  - train
  - validation
  - test

Με αυτόν τον τρόπο:

- δεν υπάρχει leakage από validation/test προς τα preprocessing statistics
- διατηρείται το ίδιο target-domain contract για όλα τα splits
- δεν αλλάζει το feature engineering scope του `NB04`

In [5]:
# ============================================================
# Train-only threshold fitting για target clipping
# ============================================================

train_target = train_df[TARGET_COLUMN]

if train_target.isnull().any():
    raise ValueError("Βρέθηκαν null values στο train target πριν το threshold fitting.")

train_mean = float(train_target.mean())
train_std = float(train_target.std(ddof=1))

if not np.isfinite(train_mean):
    raise ValueError("Το train target mean δεν είναι finite.")

if not np.isfinite(train_std) or train_std <= 0:
    raise ValueError(
        f"Μη έγκυρο train target std για Z-score clipping: {train_std}"
    )

Z_SCORE_THRESHOLD = float(getattr(cfg, "Z_SCORE_THRESHOLD", 3.0))

lower_clip = train_mean - Z_SCORE_THRESHOLD * train_std
upper_clip = train_mean + Z_SCORE_THRESHOLD * train_std

THRESHOLD_FIT_SPLIT = "train"
THRESHOLD_FIT_ROWS = len(train_df)


def compute_outlier_mask(
    series: pd.Series,
    mean_: float,
    std_: float,
    z_thr: float,
) -> pd.Series:
    z_values = ((series - mean_) / std_).abs()
    return z_values > z_thr


train_outlier_mask = compute_outlier_mask(
    train_df[TARGET_COLUMN], train_mean, train_std, Z_SCORE_THRESHOLD
)
val_outlier_mask = compute_outlier_mask(
    val_df[TARGET_COLUMN], train_mean, train_std, Z_SCORE_THRESHOLD
)
test_outlier_mask = compute_outlier_mask(
    test_df[TARGET_COLUMN], train_mean, train_std, Z_SCORE_THRESHOLD
)

threshold_summary_df = pd.DataFrame(
    {
        "metric": [
            "threshold_fit_split",
            "threshold_fit_rows",
            "train_mean",
            "train_std",
            "z_score_threshold",
            "lower_clip",
            "upper_clip",
        ],
        "value": [
            THRESHOLD_FIT_SPLIT,
            THRESHOLD_FIT_ROWS,
            train_mean,
            train_std,
            Z_SCORE_THRESHOLD,
            lower_clip,
            upper_clip,
        ],
    }
)

outlier_summary_df = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "outliers_before_clip": [
            int(train_outlier_mask.sum()),
            int(val_outlier_mask.sum()),
            int(test_outlier_mask.sum()),
        ],
        "outlier_rate_pct": [
            100.0 * float(train_outlier_mask.mean()),
            100.0 * float(val_outlier_mask.mean()),
            100.0 * float(test_outlier_mask.mean()),
        ],
    }
)

print("Train-only outlier fitting completed")
print("-" * 70)
display(threshold_summary_df)
display(outlier_summary_df)

Train-only outlier fitting completed
----------------------------------------------------------------------


,metric,value
0,threshold_fit_split,train
1,threshold_fit_rows,1982736
2,train_mean,0.173518
3,train_std,0.271611
4,z_score_threshold,3.0
5,lower_clip,-0.641316
6,upper_clip,0.988351


,split,rows,outliers_before_clip,outlier_rate_pct
0,train,1982736,69530,3.506770
1,val,182998,4718,2.578170
2,test,1086336,55852,5.141319


In [6]:
# ============================================================
# Εφαρμογή clipping με train-fitted thresholds
# Χρησιμοποιούμε .loc για pandas-safe assignment style.
# ============================================================

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    split_df.loc[:, TARGET_COLUMN] = split_df[TARGET_COLUMN].clip(
        lower=lower_clip,
        upper=upper_clip,
    )

print("Clipping applied successfully")
print("-" * 70)
print(f"Target clipping range: [{lower_clip:.6f}, {upper_clip:.6f}]")

Clipping applied successfully
----------------------------------------------------------------------
Target clipping range: [-0.641316, 0.988351]


## Τελικοί έλεγχοι ακεραιότητας και safe export

Πριν από το export, το notebook ελέγχει ρητά ότι ισχύουν όλα τα παρακάτω:

- δεν υπάρχουν null values
- δεν υπάρχουν duplicate `(park_id, timestamp)` rows
- δεν υπάρχει overlap μεταξύ splits
- η χρονική σειρά είναι monotonic ανά park
- το schema είναι identical σε train / val / test
- τα thresholds fit-αρίστηκαν μόνο στο train
- το export order είναι deterministic
- τα downstream modeling columns μπορούν να οριστούν με καθαρό τρόπο

Επιπλέον, το export γίνεται με **safe overwrite / atomic replace** και ακολουθεί
άμεσο file-level verification, ώστε να πιαστεί αμέσως οποιοδήποτε corrupted artifact.

In [7]:
# ============================================================
# Deterministic export order
# ============================================================

for split_df in [train_df, val_df, test_df]:
    split_df.sort_values(
        [PARK_ID_COLUMN, TIMESTAMP_COLUMN],
        kind="mergesort",
        inplace=True,
    )
    split_df.reset_index(drop=True, inplace=True)


# ============================================================
# Full null checks
# ============================================================

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    null_count = int(split_df.isnull().sum().sum())
    if null_count > 0:
        null_detail = split_df.isnull().sum()
        null_detail = null_detail[null_detail > 0].sort_values(ascending=False)
        raise ValueError(
            f"Βρέθηκαν null values στο split `{split_name}`:\n"
            f"{null_detail.to_string()}"
        )


# ============================================================
# Duplicate key checks
# ============================================================

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    dup_count = int(
        split_df.duplicated(subset=[PARK_ID_COLUMN, TIMESTAMP_COLUMN]).sum()
    )
    if dup_count > 0:
        raise ValueError(
            f"Βρέθηκαν duplicate ({PARK_ID_COLUMN}, {TIMESTAMP_COLUMN}) rows "
            f"στο split `{split_name}`: {dup_count}"
        )


# ============================================================
# Monotonic ordering ανά split / park
# ============================================================

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    bad_parks = []

    for park_id, park_slice in split_df.groupby(PARK_ID_COLUMN, sort=False):
        if not park_slice[TIMESTAMP_COLUMN].is_monotonic_increasing:
            bad_parks.append(park_id)

    if bad_parks:
        raise ValueError(
            f"Βρέθηκαν parks με μη monotonic ordering στο split `{split_name}`: "
            f"{bad_parks[:10]}"
        )


# ============================================================
# Same schema across splits
# ============================================================

reference_columns = train_df.columns.tolist()

if val_df.columns.tolist() != reference_columns:
    raise ValueError("Το validation schema δεν είναι identical με το train schema.")

if test_df.columns.tolist() != reference_columns:
    raise ValueError("Το test schema δεν είναι identical με το train schema.")


# ============================================================
# Train-only threshold fitting check
# ============================================================

if THRESHOLD_FIT_SPLIT != "train":
    raise ValueError("Τα clipping thresholds δεν fit-αρίστηκαν στο train split.")

if THRESHOLD_FIT_ROWS != len(train_df):
    raise ValueError(
        "Το πλήθος rows που δηλώθηκε για threshold fitting δεν συμφωνεί με το train split."
    )


# ============================================================
# Target range post-clipping checks
# ============================================================

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    split_min = float(split_df[TARGET_COLUMN].min())
    split_max = float(split_df[TARGET_COLUMN].max())

    if split_min < lower_clip - 1e-12 or split_max > upper_clip + 1e-12:
        raise ValueError(
            f"Το split `{split_name}` παραβιάζει το clipping range. "
            f"Observed=[{split_min}, {split_max}], "
            f"Expected=[{lower_clip}, {upper_clip}]"
        )


# ============================================================
# Flag contract checks
# ============================================================

expected_flag_sets = {
    "train": {0},
    "val": {0},
    "test": {1},
}

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    observed_flags = set(split_df[TEST_FLAG_COLUMN].dropna().astype(int).unique().tolist())
    if observed_flags != expected_flag_sets[split_name]:
        raise ValueError(
            f"Λάθος test_flag contract στο split `{split_name}`. "
            f"Observed={sorted(observed_flags)}, Expected={sorted(expected_flag_sets[split_name])}"
        )


# ============================================================
# Downstream guardrail:
# Τα blocked columns δεν πρέπει by default να είναι numeric modeling features.
# ============================================================

for blocked_col in CONTROL_IDENTIFIER_COLUMNS + PROVENANCE_AUDIT_COLUMNS:
    if blocked_col in NUMERIC_MODELING_FEATURES:
        raise ValueError(
            f"Η blocked στήλη `{blocked_col}` δεν πρέπει να εμφανίζεται "
            "στα NUMERIC_MODELING_FEATURES."
        )


# ============================================================
# Βοηθητικές συναρτήσεις για safe export και verification
# ============================================================

def count_data_rows_fast(csv_path: Path, chunk_size: int = 8 * 1024 * 1024) -> int:
    """
    Μετρά ακριβώς τα data rows χωρίς να φορτώσει όλο το CSV στη μνήμη.
    Αφαιρεί τη header γραμμή.
    """
    newline_count = 0

    with open(csv_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")

        if f.tell() == 0:
            total_lines = 0
        else:
            f.seek(-1, 2)
            ends_with_newline = f.read(1) == b"\n"
            total_lines = newline_count if ends_with_newline else newline_count + 1

    return max(total_lines - 1, 0)


def verify_export_file(
    csv_path: Path,
    expected_rows: int,
    expected_cols: int,
    required_columns: set[str],
    expected_flag_values: set[int],
) -> dict:
    """
    File-level verification αμέσως μετά το export.
    Αν κάτι είναι λάθος, σταματάμε το notebook.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Το export δεν βρέθηκε: {csv_path}")

    header_df = pd.read_csv(csv_path, nrows=0)
    actual_columns = header_df.columns.tolist()
    actual_col_count = len(actual_columns)

    if actual_col_count != expected_cols:
        raise ValueError(
            f"Λάθος column count στο {csv_path.name}. "
            f"Observed={actual_col_count}, Expected={expected_cols}"
        )

    missing_required = sorted(required_columns - set(actual_columns))
    if missing_required:
        raise ValueError(
            f"Λείπουν required columns στο {csv_path.name}: {missing_required}"
        )

    actual_row_count = count_data_rows_fast(csv_path)
    if actual_row_count != expected_rows:
        raise ValueError(
            f"Λάθος row count στο {csv_path.name}. "
            f"Observed={actual_row_count:,}, Expected={expected_rows:,}"
        )

    sample_df = pd.read_csv(csv_path, nrows=5)
    if sample_df.shape[1] != expected_cols:
        raise ValueError(
            f"Το sample parse του {csv_path.name} επέστρεψε λάθος πλήθος στηλών: "
            f"{sample_df.shape[1]} αντί για {expected_cols}"
        )

    observed_flags = set(
        pd.read_csv(csv_path, usecols=[TEST_FLAG_COLUMN], nrows=50_000)[TEST_FLAG_COLUMN]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if observed_flags != expected_flag_values:
        raise ValueError(
            f"Λάθος flag set στο {csv_path.name}. "
            f"Observed={sorted(observed_flags)}, Expected={sorted(expected_flag_values)}"
        )

    return {
        "file": csv_path.name,
        "rows": actual_row_count,
        "columns": actual_col_count,
        "flags": sorted(observed_flags),
        "path": str(csv_path),
    }


def safe_export_csv(df_to_export: pd.DataFrame, final_path: Path) -> None:
    """
    Safe export:
    - διαγράφει stale temp file αν υπάρχει
    - γράφει πρώτα σε temporary path
    - μετά κάνει atomic replace στο canonical path
    """
    final_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = final_path.with_suffix(final_path.suffix + ".tmp")

    if tmp_path.exists():
        tmp_path.unlink()

    if final_path.exists():
        final_path.unlink()

    df_to_export.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
        date_format="%Y-%m-%d %H:%M:%S",
        quoting=csv.QUOTE_MINIMAL,
    )

    tmp_path.replace(final_path)


# ============================================================
# Canonical export paths
# ============================================================

train_path = DATA_PROCESSED / "train_final.csv"
val_path = DATA_PROCESSED / "val_final.csv"
test_path = DATA_PROCESSED / "test_final.csv"


# ============================================================
# Safe exports
# ============================================================

safe_export_csv(train_df, train_path)
safe_export_csv(val_df, val_path)
safe_export_csv(test_df, test_path)


# ============================================================
# Άμεσο verification των exports
# Χρησιμοποιούμε ως expected_rows τα in-memory splits.
# ============================================================

required_export_columns = {
    PARK_ID_COLUMN,
    TIMESTAMP_COLUMN,
    TEST_FLAG_COLUMN,
    TARGET_COLUMN,
    BASELINE_COLUMN,
}

verification_results = [
    verify_export_file(
        train_path,
        expected_rows=len(train_df),
        expected_cols=len(train_df.columns),
        required_columns=required_export_columns,
        expected_flag_values={0},
    ),
    verify_export_file(
        val_path,
        expected_rows=len(val_df),
        expected_cols=len(val_df.columns),
        required_columns=required_export_columns,
        expected_flag_values={0},
    ),
    verify_export_file(
        test_path,
        expected_rows=len(test_df),
        expected_cols=len(test_df.columns),
        required_columns=required_export_columns,
        expected_flag_values={1},
    ),
]

export_summary_df = pd.DataFrame(verification_results)

downstream_contract_df = pd.DataFrame(
    {
        "group": [
            "target",
            "control_identifier",
            "provenance_audit",
            "candidate_modeling_features",
            "numeric_modeling_features",
        ],
        "columns": [
            TARGET_COLUMNS,
            CONTROL_IDENTIFIER_COLUMNS,
            PROVENANCE_AUDIT_COLUMNS,
            CANDIDATE_MODELING_FEATURES,
            NUMERIC_MODELING_FEATURES,
        ],
    }
)

print("NB05 exports completed successfully")
print("-" * 70)
display(export_summary_df)

print("Downstream column contract")
print("-" * 70)
display(downstream_contract_df)

NB05 exports completed successfully
----------------------------------------------------------------------


,file,rows,columns,flags,path
0,train_final.csv,1982736,47,[0],C:\Users\diony\Desktop\WindPower_DigitalTwin\d...
1,val_final.csv,182998,47,[0],C:\Users\diony\Desktop\WindPower_DigitalTwin\d...
2,test_final.csv,1086336,47,[1],C:\Users\diony\Desktop\WindPower_DigitalTwin\d...


Downstream column contract
----------------------------------------------------------------------


,group,columns
0,target,[Power_Output_Normalized]
1,control_identifier,"[park_id, timestamp]"
2,provenance_audit,"[test_flag, Baseline_Prediction]"
3,candidate_modeling_features,"[turbine, hub_height_m, rotor_diameter_m, nomi..."
4,numeric_modeling_features,"[hub_height_m, rotor_diameter_m, nominal_power..."


## Συμπέρασμα

Το `NB05` ολοκλήρωσε το canonical stage για:

- **flag-aware temporal split**
- **train-only outlier fitting**
- **deterministic export**
- **ρητό column-role contract** για τα downstream notebooks

## Τι εγγυάται πλέον το notebook

- strict consumption του canonical `NB04` export
- no raw reparsing
- no split overlap
- no duplicate backbone keys
- monotonic temporal ordering
- identical schema across `train / val / test`
- clipping thresholds fitted μόνο στο `train`
- explicit separation μεταξύ:
  - target
  - control / identifier columns
  - candidate modeling features
  - provenance / audit columns

## Downstream rule για NB06 / NB07

Τα `NB06` και `NB07` δεν πρέπει να χρησιμοποιούν ως model inputs:

- `park_id`
- `timestamp`
- `test_flag`
- `Baseline_Prediction`

Επίσης, non-numeric helper metadata όπως το `turbine` δεν πρέπει να χρησιμοποιούνται
χωρίς explicit encoding policy.

## Practical note

Μην ανοίξεις και μην ξανασώσεις τα exported CSV μέσω Excel πριν από το verification.
Το canonical check πρέπει να γίνεται πάνω στα αρχεία όπως γράφτηκαν από το notebook.

In [8]:
# ============================================================
# Quick post-export verification για τα canonical NB05 artifacts
# ============================================================

expected = {
    "train_final.csv": {"rows": 1_982_736, "cols": 47, "flags": {0}},
    "val_final.csv":   {"rows":   182_998, "cols": 47, "flags": {0}},
    "test_final.csv":  {"rows": 1_086_336, "cols": 47, "flags": {1}},
}

required_columns = {
    "park_id",
    "timestamp",
    "test_flag",
    "Power_Output_Normalized",
    "Baseline_Prediction",
}


def mark(ok: bool) -> str:
    return "PASS" if ok else "FAIL"


def count_data_rows_fast(csv_path: Path, chunk_size: int = 8 * 1024 * 1024) -> int:
    newline_count = 0

    with open(csv_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            newline_count += chunk.count(b"\n")

        if f.tell() == 0:
            total_lines = 0
        else:
            f.seek(-1, 2)
            ends_with_newline = f.read(1) == b"\n"
            total_lines = newline_count if ends_with_newline else newline_count + 1

    return max(total_lines - 1, 0)


results = []
all_ok = True

for filename, spec in expected.items():
    csv_path = DATA_PROCESSED / filename

    print("=" * 78)
    print(f"Έλεγχος αρχείου: {filename}")
    print(f"Path: {csv_path}")

    # Header-only read
    try:
        header_df = pd.read_csv(csv_path, nrows=0)
        actual_columns = list(header_df.columns)
        actual_col_count = len(actual_columns)
        col_count_ok = actual_col_count == spec["cols"]
    except Exception as e:
        actual_columns = []
        actual_col_count = None
        col_count_ok = False
        print(f"[FAIL] Αποτυχία ανάγνωσης header: {e}")

    # Exact row count
    try:
        actual_row_count = count_data_rows_fast(csv_path)
        row_count_ok = actual_row_count == spec["rows"]
    except Exception as e:
        actual_row_count = None
        row_count_ok = False
        print(f"[FAIL] Αποτυχία καταμέτρησης γραμμών: {e}")

    # Sample parse
    try:
        sample_df = pd.read_csv(csv_path, nrows=5)
        sample_parse_ok = sample_df.shape[1] == spec["cols"]
    except Exception as e:
        sample_df = None
        sample_parse_ok = False
        print(f"[FAIL] Αποτυχία sample parse: {e}")

    # Required columns
    has_required_columns = required_columns.issubset(set(actual_columns))

    # Flag check
    try:
        observed_flags = set()
        for chunk in pd.read_csv(csv_path, usecols=["test_flag"], chunksize=200_000):
            observed_flags.update(chunk["test_flag"].dropna().astype(int).unique().tolist())
        flags_ok = observed_flags == spec["flags"]
    except Exception as e:
        observed_flags = set()
        flags_ok = False
        print(f"[FAIL] Αποτυχία flag check: {e}")

    file_ok = (
        row_count_ok
        and col_count_ok
        and sample_parse_ok
        and has_required_columns
        and flags_ok
    )
    all_ok = all_ok and file_ok

    print(f"Αναμενόμενα rows: {spec['rows']:,}")
    print(f"Πραγματικά rows:  {actual_row_count:,}" if actual_row_count is not None else "Πραγματικά rows:  <ERROR>")
    print(f"Rows check:       {mark(row_count_ok)}")

    print(f"Αναμενόμενες cols: {spec['cols']}")
    print(f"Πραγματικές cols:  {actual_col_count}" if actual_col_count is not None else "Πραγματικές cols:  <ERROR>")
    print(f"Cols check:        {mark(col_count_ok)}")

    print(f"Sample parse check:     {mark(sample_parse_ok)}")
    print(f"Required columns check: {mark(has_required_columns)}")
    print(f"Flags check:            {mark(flags_ok)}")
    print(f"Observed flags:         {sorted(observed_flags) if observed_flags else []}")
    print(f"Συνολικό αποτέλεσμα:    {mark(file_ok)}")

    results.append(
        {
            "file": filename,
            "expected_rows": spec["rows"],
            "actual_rows": actual_row_count,
            "rows_ok": row_count_ok,
            "expected_cols": spec["cols"],
            "actual_cols": actual_col_count,
            "cols_ok": col_count_ok,
            "sample_parse_ok": sample_parse_ok,
            "required_columns_ok": has_required_columns,
            "flags_ok": flags_ok,
            "observed_flags": sorted(observed_flags) if observed_flags else [],
            "file_ok": file_ok,
        }
    )

summary_df = pd.DataFrame(results)

print("\n" + "=" * 78)
print("ΤΕΛΙΚΟ SUMMARY")
display(summary_df)

if all_ok:
    print("\n✅ QUICK VERIFICATION PASSED")
    print("Τα exported split artifacts είναι συνεπή με το current canonical NB05 rerun state.")
else:
    print("\n❌ QUICK VERIFICATION FAILED")
    print("Το current NB05 export state ΔΕΝ είναι ακόμα ασφαλές για push.")
    print("Χρειάζεται έλεγχος / rerun του export logic πριν από commit.")

Έλεγχος αρχείου: train_final.csv
Path: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\train_final.csv
Αναμενόμενα rows: 1,982,736
Πραγματικά rows:  1,982,736
Rows check:       PASS
Αναμενόμενες cols: 47
Πραγματικές cols:  47
Cols check:        PASS
Sample parse check:     PASS
Required columns check: PASS
Flags check:            PASS
Observed flags:         [0]
Συνολικό αποτέλεσμα:    PASS
Έλεγχος αρχείου: val_final.csv
Path: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\val_final.csv
Αναμενόμενα rows: 182,998
Πραγματικά rows:  182,998
Rows check:       PASS
Αναμενόμενες cols: 47
Πραγματικές cols:  47
Cols check:        PASS
Sample parse check:     PASS
Required columns check: PASS
Flags check:            PASS
Observed flags:         [0]
Συνολικό αποτέλεσμα:    PASS
Έλεγχος αρχείου: test_final.csv
Path: C:\Users\diony\Desktop\WindPower_DigitalTwin\data\processed\test_final.csv
Αναμενόμενα rows: 1,086,336
Πραγματικά rows:  1,086,336
Rows check:       PASS
Αναμ

,file,expected_rows,actual_rows,rows_ok,expected_cols,actual_cols,cols_ok,sample_parse_ok,required_columns_ok,flags_ok,observed_flags,file_ok
0,train_final.csv,1982736,1982736,True,47,47,True,True,True,True,[0],True
1,val_final.csv,182998,182998,True,47,47,True,True,True,True,[0],True
2,test_final.csv,1086336,1086336,True,47,47,True,True,True,True,[1],True



✅ QUICK VERIFICATION PASSED
Τα exported split artifacts είναι συνεπή με το current canonical NB05 rerun state.
